# Semana 6: Redes Neuronales para NLP y Embeddings Entrenables
## Notebook de Ejercicios (NB2) – Clasificación de Sentimiento con PyTorch

**Propósito:** Aplicar redes neuronales con embeddings entrenables a un problema real de análisis de sentimiento sobre reseñas de películas de IMDb, y comparar con modelos clásicos.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Crear DataLoader con padding para manejar longitudes variables.
- Entrenar un modelo de promedio de embeddings con PyTorch.
- Evaluar en test y comparar con modelos clásicos (Naive Bayes, SVM).
- Experimentar con dropout y regularización.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos PyTorch.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Scikit-learn para modelos clásicos y métricas
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Hugging Face datasets
try:
    from datasets import load_dataset
    DATASETS_AVAILABLE = True
except ImportError:
    DATASETS_AVAILABLE = False
    print("Nota: datasets no está instalado. Se instalará más adelante.")

# NLTK para tokenización
import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize

# Configuración de visualización
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Configuración de PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Semillas para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

print("\nLibrerías importadas correctamente.")

---
## 1. Carga y Preprocesamiento del Dataset IMDb

Cargamos un subconjunto balanceado de IMDb reviews.

In [ ]:
if not DATASETS_AVAILABLE:
    !pip install datasets
    from datasets import load_dataset

# Cargamos el dataset IMDb
print("Cargando dataset IMDb...")
imdb = load_dataset('imdb')

# Usamos un subconjunto balanceado para el ejercicio
train_size = 2000  # 2000 reseñas para entrenamiento
test_size = 500    # 500 para prueba

# Tomar muestras balanceadas
train_texts = imdb['train']['text'][:train_size]
train_labels = imdb['train']['label'][:train_size]

test_texts = imdb['test']['text'][:test_size]
test_labels = imdb['test']['label'][:test_size]

print(f"Entrenamiento: {len(train_texts)} reseñas")
print(f"Prueba: {len(test_texts)} reseñas")
print(f"\nProporción de positivos en entrenamiento: {np.mean(train_labels):.2%}")
print(f"Proporción de positivos en prueba: {np.mean(test_labels):.2%}")

# Mostrar un ejemplo
print("\n=== Ejemplo de reseña ===")
sentimiento = "POSITIVO" if train_labels[0] == 1 else "NEGATIVO"
print(f"Sentimiento: {sentimiento}")
print(f"Texto: {train_texts[0][:500]}...")

---
## 2. Construcción del Vocabulario y Tokenización

### 2.1. Crear vocabulario a partir de los datos de entrenamiento

In [ ]:
def tokenize(text):
    """Tokenización simple por palabras."""
    return text.lower().split()

# Construir vocabulario
def build_vocab(texts, max_size=10000, min_freq=2):
    """Construye vocabulario con tokens especiales."""
    word_counts = Counter()
    for text in texts:
        word_counts.update(tokenize(text))
    
    vocab = {'<PAD>': 0, '<UNK>': 1}
    
    # Añadir palabras que cumplen frecuencia mínima
    for word, count in word_counts.items():
        if count >= min_freq and len(vocab) < max_size:
            vocab[word] = len(vocab)
    
    return vocab

vocab = build_vocab(train_texts, max_size=10000, min_freq=2)
print(f"Tamaño del vocabulario: {len(vocab)}")
print(f"Primeras 10 palabras: {list(vocab.keys())[:10]}")

### 2.2. Convertir textos a índices con padding

In [ ]:
def encode_texts(texts, vocab, max_len=200):
    """Convierte textos a secuencias de índices con padding."""
    encoded = []
    for text in texts:
        tokens = tokenize(text)[:max_len]
        indices = [vocab.get(token, vocab['<UNK>']) for token in tokens]
        # Padding
        indices += [vocab['<PAD>']] * (max_len - len(indices))
        encoded.append(indices)
    return torch.tensor(encoded)

max_len = 200  # longitud máxima de secuencia

X_train = encode_texts(train_texts, vocab, max_len)
y_train = torch.tensor(train_labels)

X_test = encode_texts(test_texts, vocab, max_len)
y_test = torch.tensor(test_labels)

print(f"Forma de X_train: {X_train.shape}")
print(f"Forma de X_test: {X_test.shape}")

---
## 3. Creación de DataLoader con Padding

Creamos DataLoader para iterar sobre los datos en lotes.

In [ ]:
batch_size = 32

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Número de lotes de entrenamiento: {len(train_loader)}")
print(f"Número de lotes de prueba: {len(test_loader)}")

# Verificar un lote
for batch_X, batch_y in train_loader:
    print(f"Batch X shape: {batch_X.shape}")
    print(f"Batch y shape: {batch_y.shape}")
    break

---
## 4. Definición del Modelo de Promedio de Embeddings

Creamos un modelo con:
- Capa de embedding entrenable
- Promedio de embeddings (ignorando padding)
- MLP con dropout

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=64, num_classes=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_dim)
        
        # Crear máscara para tokens que no son padding
        mask = (x != 0).unsqueeze(-1).float()  # (batch_size, seq_len, 1)
        
        # Suma de embeddings con máscara
        sum_embeddings = (embedded * mask).sum(dim=1)  # (batch_size, embed_dim)
        # Número de tokens reales por secuencia
        lengths = mask.sum(dim=1)  # (batch_size, 1)
        # Evitar división por cero
        pooled = sum_embeddings / (lengths + 1e-8)  # (batch_size, embed_dim)
        
        h = self.dropout(self.relu(self.fc1(pooled)))
        out = self.fc2(h)
        return out

# Instanciar modelo
model = SentimentClassifier(len(vocab), embed_dim=100, hidden_dim=64, dropout=0.5)
model.to(device)

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total de parámetros: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")

---
## 5. Entrenamiento del Modelo

### 5.1. Función de entrenamiento

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_X, batch_y in loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    
    return total_loss / len(loader), correct / total

In [ ]:
epochs = 10
train_losses = []
train_accs = []
test_losses = []
test_accs = []

print("Iniciando entrenamiento...")
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    print(f"Epoch {epoch+1:2d}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} - Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

print("\nEntrenamiento completado.")

### 5.2. Visualización de la evolución del entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train')
axes[0].plot(test_losses, label='Test')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida')
axes[0].set_title('Evolución de la pérdida')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(train_accs, label='Train')
axes[1].plot(test_accs, label='Test')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Evolución del accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

final_test_acc = test_accs[-1]
print(f"Accuracy final en test: {final_test_acc:.4f}")

---
## 6. Comparación con Modelos Clásicos (Naive Bayes y SVM)

Entrenamos modelos clásicos para comparar con nuestro modelo neuronal.

In [ ]:
# Vectorizador TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1,2))

# Transformar textos
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_test_tfidf = tfidf_vectorizer.transform(test_texts)

# Naive Bayes
print("Entrenando Naive Bayes...")
start_nb = time.time()
nb = MultinomialNB(alpha=1.0)
nb.fit(X_train_tfidf, train_labels)
nb_time = time.time() - start_nb
nb_pred = nb.predict(X_test_tfidf)
nb_acc = accuracy_score(test_labels, nb_pred)

# SVM Lineal
print("Entrenando SVM Lineal...")
start_svm = time.time()
svm = LinearSVC(random_state=42, C=1.0, max_iter=2000)
svm.fit(X_train_tfidf, train_labels)
svm_time = time.time() - start_svm
svm_pred = svm.predict(X_test_tfidf)
svm_acc = accuracy_score(test_labels, svm_pred)

print("\n=== RESULTADOS MODELOS CLÁSICOS ===")
print(f"Naive Bayes - Accuracy: {nb_acc:.4f}, Tiempo: {nb_time:.2f}s")
print(f"SVM - Accuracy: {svm_acc:.4f}, Tiempo: {svm_time:.2f}s")

### 6.1. Comparación de resultados

In [ ]:
comparacion = pd.DataFrame({
    'Modelo': ['Naive Bayes', 'SVM', 'Red Neuronal (PyTorch)'],
    'Accuracy': [nb_acc, svm_acc, final_test_acc],
    'Tiempo (s)': [nb_time, svm_time, epochs * len(train_loader) * 0.01]  # estimación
})

print("=== COMPARACIÓN DE MODELOS ===")
print(comparacion.round(4))

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(comparacion['Modelo'], comparacion['Accuracy'])
plt.xlabel('Modelo')
plt.ylabel('Accuracy')
plt.title('Comparación de Accuracy entre Modelos')
plt.ylim(0.7, 0.9)
for i, v in enumerate(comparacion['Accuracy']):
    plt.text(i, v + 0.005, f'{v:.4f}', ha='center')
plt.show()

---
## 7. Experimentación con Dropout y Regularización

Probamos diferentes valores de dropout y weight decay para ver cómo afectan al rendimiento.

In [ ]:
def train_with_params(dropout=0.5, weight_decay=1e-4, epochs=5):
    """Entrena modelo con parámetros específicos."""
    model = SentimentClassifier(len(vocab), embed_dim=100, hidden_dim=64, dropout=dropout)
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)
    
    for epoch in range(epochs):
        train_epoch(model, train_loader, optimizer, criterion)
    
    _, test_acc = evaluate(model, test_loader, criterion)
    return test_acc

print("Experimentando con diferentes configuraciones...")

configs = [
    (0.3, 1e-4),
    (0.5, 1e-4),
    (0.7, 1e-4),
    (0.5, 1e-3),
    (0.5, 1e-5)
]

results_exp = []
for dropout, wd in configs:
    acc = train_with_params(dropout, wd)
    results_exp.append({
        'dropout': dropout,
        'weight_decay': wd,
        'test_accuracy': acc
    })
    print(f"dropout={dropout}, weight_decay={wd} -> accuracy={acc:.4f}")

df_exp = pd.DataFrame(results_exp)
print("\n=== RESULTADOS EXPERIMENTACIÓN ===")
print(df_exp)

---
## 8. Conclusiones

En este notebook hemos aplicado redes neuronales a un problema real de análisis de sentimiento:

✔️ **DataLoader con padding**: Creamos DataLoader eficiente para manejar longitudes variables.

✔️ **Modelo de promedio de embeddings**: Implementamos y entrenamos un modelo con PyTorch.

✔️ **Comparación con modelos clásicos**:
- Naive Bayes y SVM obtienen buenos resultados (≈85-88% accuracy).
- La red neuronal puede igualar o superar a los clásicos con suficiente entrenamiento.

✔️ **Experimentación con regularización**:
- Dropout ayuda a prevenir sobreajuste (valores típicos 0.3-0.5).
- Weight decay también contribuye a la regularización.

**Lección clave**: Las redes neuronales con embeddings entrenables son una alternativa competitiva a los modelos clásicos, especialmente cuando se dispone de suficientes datos. La regularización es crucial para evitar sobreajuste.

---
**Fin del Notebook de Ejercicios - Semana 6 (NLP)**